# Reversible candidates

Shows measured pilot development losses and trained-checkpoint numerical gates.

In [1]:
from pathlib import Path
import json, csv, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src' / 'revllm').exists(), 'Execute with repository root as working directory'
sys.path.insert(0, str(ROOT / 'src'))
from revllm.data import digest_file
def read_final(path):
    result = json.loads(path.read_text())
    updates = [json.loads(line) for line in (path.parent/'metrics.jsonl').read_text().splitlines()]
    assert sum(row['valid_targets'] for row in updates if row['status']=='COMMITTED') == result['committed_targets']
    return result


In [2]:
# Real two-update execution; excluded from assignment token totals.
from tempfile import TemporaryDirectory
from revllm.model import ModelConfig
from revllm.train import RunConfig, train
with TemporaryDirectory(prefix='revllm-notebook-') as scratch:
    cfg = RunConfig(run_id='notebook-demo',
                    model=ModelConfig(n_layer=2, n_embd=16, n_head=2, block_size=16,
                                      integrator='coupled_euler', backward_mode='reconstructed', step_size=0.5),
                    data_dir=ROOT/'data', out_dir=Path(scratch), physical_batch=2, effective_batch=2,
                    target_tokens=256, warmup_tokens=128, use_amp=False)
    measured = train(cfg, max_updates=2)
    assert measured['committed_targets'] == 64
    print({'demonstration': 'coupled_euler', 'successful_updates': measured['updates'],
           'committed_targets': measured['committed_targets'], 'assignment_run': False})


{'demonstration': 'coupled_euler', 'successful_updates': 2, 'committed_targets': 64, 'assignment_run': False}


In [3]:
base = ROOT/'experiments'/'kaggle-pilots-v3'
rows = []
for path in sorted((base/'runs').glob('pilot-*/final.json')):
    final = json.loads(path.read_text())
    ev = json.loads((path.parent/'eval.jsonl').read_text().splitlines()[-1])
    rows.append((final['run_id'], ev['dev_loss_nats'], final['committed_targets']))
for row in sorted(rows, key=lambda x:x[1]): print(row)

('pilot-coupled050', 3.9153412096202374, 2000000)
('pilot-coupled100', 3.9274092819541693, 2000000)
('pilot-coupled025', 3.9403002094477415, 2000000)
('pilot-conventional', 3.9543055798858404, 2000000)
('pilot-midpoint050', 3.9630864430218935, 2000000)
('pilot-midpoint025', 3.963239347562194, 2000000)
('pilot-blended025', 3.9665831681340933, 2000000)
('pilot-midpoint100', 3.9732298869639635, 2000000)


In [4]:
checks = json.loads((base/'trained_checkpoint_probes.json').read_text())
print([(c['run_id'],c['status'],c.get('global_relative_gradient_error_fp32'),c.get('reconstruction',{}).get('relative_reconstruction_error')) for c in checks])

[('pilot-blended025', 'PASS', 1.751029776642099e-05, 9.430999853066169e-06), ('pilot-conventional', 'NOT_REVERSIBLE', None, None), ('pilot-coupled025', 'PASS', 9.521692732050724e-07, 5.277275505477519e-08), ('pilot-coupled050', 'PASS', 1.5289439261323423e-06, 7.59929505989021e-08), ('pilot-coupled100', 'PASS', 4.891994649369735e-06, 1.213454510207157e-07), ('pilot-midpoint025', 'PASS', 4.7566962280143343e-07, 5.2847521914145545e-08), ('pilot-midpoint050', 'PASS', 5.740029678236169e-07, 7.972771953745905e-08), ('pilot-midpoint100', 'PASS', 4.211333646253479e-07, 1.5591518831570284e-07)]


In [5]:
for campaign in ('kaggle-full-v1','kaggle-locked-v1','kaggle-locked-repair-v1'):
    for path in sorted((ROOT/'experiments'/campaign/'runs').glob('[BCDE]-*/final.json')):
        r=read_final(path)
        print(r['run_id'], r['committed_targets'], r['holdout_loss_nats'])

B-1337 50000000 2.750823995727539
B-2027 50000000 2.774702043701172
C-1337 50000000 3.092301239868164
C-2027 50000000 3.106536435546875
D-1337 50000000 2.748128696533203
E-1337 50000000 3.092173278564453
B-L-1337 50000000 2.742739299560547
D-L-1337 50000000 2.7440332314453126
B-L-2027 50000000 2.773651153076172
C-L191-1337 50000000 3.0717848221435546
C-L191-2027 50000000 3.095928124145508
E-L191-1337 50000000 3.072389580078125
